# Data Transformation – Gold Layer
## fact_financial_transaction

This notebook performs the **dimensional modeling transformations** that populate the **gold layer**, generating **business-ready tables** such as dimensions and fact tables in **Delta Lake** format.

In [0]:
CREATE OR REPLACE TABLE gold.take_home_test.fact_financial_transaction AS
-- =========================================================
-- 1) Base unified (silver)
-- =========================================================
WITH unified AS (
    SELECT 
      transaction_id,
      account_id,
      amount,
      requested_time_id,
      completed_time_id,
      is_success,
      transaction_method,
      movement_type
    FROM silver.take_home_test.transactions
),

-- =========================================================
-- 2) Add timestamps from d_time
-- =========================================================
enriched_time AS (
    SELECT
        u.*,
        tr.event_timestamp AS requested_at,
        tc.event_timestamp AS completed_at,
        CAST(date_format(to_timestamp(tr.event_timestamp), 'yyyyMMdd') AS INT) AS requested_date_id,
        CAST(date_format(to_timestamp(tc.event_timestamp), 'yyyyMMdd') AS INT) AS completed_date_id
    FROM unified u
    LEFT JOIN silver.take_home_test.d_time tr
        ON u.requested_time_id = tr.time_id
    LEFT JOIN silver.take_home_test.d_time tc
        ON u.completed_time_id = tc.time_id
),

-- =========================================================
-- 3) Enrich with account (to get customer_id)
-- =========================================================
enriched_acc AS (
    SELECT 
        et.*,
        a.customer_id
    FROM enriched_time et
    LEFT JOIN silver.take_home_test.accounts a
        ON et.account_id = a.account_id
),

-- =========================================================
-- 4) Enrich with customer (to get city_id)
-- =========================================================
enriched_customer AS (
    SELECT
        ea.*,
        c.city_id
    FROM enriched_acc ea
    LEFT JOIN silver.take_home_test.customers c
        ON ea.customer_id = c.customer_id
),

-- =========================================================
-- 5) Enrich with city (to get state_id)
-- =========================================================
enriched_city AS (
    SELECT
        ec.*,
        ci.state_id
    FROM enriched_customer ec
    LEFT JOIN silver.take_home_test.city ci
        ON ec.city_id = ci.city_id
),

-- =========================================================
-- 6) Enrich with state (to get country_id)
-- =========================================================
enriched_state AS (
    SELECT
        eci.*,
        s.country_id
    FROM enriched_city eci
    LEFT JOIN silver.take_home_test.state s
        ON eci.state_id = s.state_id
),

-- =========================================================
-- 7) Map all dimension SKs
-- =========================================================
map_dim AS (
    SELECT
        es.*,
        
        -- transaction type SK
        dtt.transaction_type_sk,

        -- account SK
        da.account_sk,

        -- customer SK
        dc.customer_sk,

        -- city SK
        dci.city_sk,

        -- state SK
        ds.state_sk,

        -- country SK
        dco.country_sk,

        -- product SK (default: Transaction)
        dp.product_sk,

        -- currency SK (default: BRL)
        dcur.currency_sk,

        -- date SKs
        dr.date_sk  AS requested_date_sk,
        dc2.date_sk AS completed_date_sk

    FROM enriched_state es
        
        LEFT JOIN gold.take_home_test.dim_transaction_type dtt
            ON es.transaction_method = dtt.transaction_method
           AND es.movement_type     = dtt.movement_type

        LEFT JOIN gold.take_home_test.dim_account da
            ON es.account_id = da.account_id

        LEFT JOIN gold.take_home_test.dim_customer dc
            ON es.customer_id = dc.customer_id

        LEFT JOIN gold.take_home_test.dim_city dci
            ON es.city_id = dci.city_id

        LEFT JOIN gold.take_home_test.dim_state ds
            ON es.state_id = ds.state_id

        LEFT JOIN gold.take_home_test.dim_country dco
            ON es.country_id = dco.country_id

        LEFT JOIN gold.take_home_test.dim_product dp
            ON dp.product_name = 'Financial Transaction'

        LEFT JOIN gold.take_home_test.dim_currency dcur
            ON dcur.currency_code = 'BRL'

        LEFT JOIN gold.take_home_test.dim_date dr
            ON es.requested_date_id = dr.date_id
        
        LEFT JOIN gold.take_home_test.dim_date dc2
            ON es.completed_date_id = dc2.date_id
)

-- =========================================================
-- 8) Final fact output
-- =========================================================
SELECT
    transaction_id,
    product_sk,
    transaction_type_sk,
    currency_sk,
    customer_sk,
    account_sk,
    country_sk,
    state_sk,
    city_sk,
    requested_date_sk,
    completed_date_sk,
    amount,
    is_success,

    -- Correct timestamps from d_time
    requested_at,
    completed_at,

    current_timestamp() AS inserted_at
FROM map_dim;
